<a href="https://colab.research.google.com/github/medealionwallis-commits/muse-s-repo/blob/main/Copy_of_Audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemini API: Audio Quickstart (Recreated)

This notebook demonstrates how to prompt Gemini Flash using audio files, including live recording and processing with a specific AI persona.

## Setup

### Install dependencies

import kagglehub path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-12b/2") print("Path to model files:", path)

In [ ]:
%pip install -U -q "google-genai>=2.9.0"

In [ ]:
%%bash
# Install pydub and its dependency pyaudioop
pip install -Uq pydub pyaudioop

In [ ]:
import os

# Install ffmpeg, a dependency for pydub to handle audio files
if not os.path.exists('/usr/bin/ffmpeg'):
    !apt-get update -qq
    !apt-get install -qq ffmpeg
    print('ffmpeg installed.')
else:
    print('ffmpeg is already installed.')

### Configure your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for a walkthrough.

In [ ]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

Select the model you want to use:

In [ ]:
MODEL_ID = "gemini-3.6-flash" # @param ["gemini-3.5-flash-lite", "gemini-2.5-flash", "gemini-3.6-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

### Upload an audio file with the File API

To use an audio file in your prompt, you must first upload it using the [File API](../quickstarts/File_API.ipynb).

In [ ]:
URL = "https://storage.googleapis.com/generativeai-downloads/data/State_of_the_Union_Address_30_January_1961.mp3"

In [ ]:
!wget -q $URL -O sample.mp3

In [ ]:
your_audio_file = client.files.upload(file='sample.mp3')

## Use the file in your prompt

In [ ]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "Listen carefully to the following audio file. Provide a brief summary."},
        {"type": "audio", "uri": your_audio_file.uri},
    ],
)

print(interaction.steps[-1].content[0].text)

## Inline Audio

For small requests, you can inline the audio data into the request, like you can with images. The audio data needs to be encoded in base64 to be sent with the prompt. Use for example PyDub to trim the first 10s of the audio:

In [ ]:
from pydub import AudioSegment
import base64

sound = AudioSegment.from_mp3("sample.mp3")
# Trim to 10 seconds (10000 ms)
trimmed_sound = sound[:10000]

audio_data = base64.b64encode(trimmed_sound.export(format="mp3").read()).decode('utf-8')

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "Describe this audio clip"},
        {"type": "audio", "data": audio_data, "mime_type": "audio/mp3"},
    ],
)

print(interaction.steps[-1].content[0].text)

## Live Audio Recording with Persona Integration

This section enables recording audio directly from your microphone and processing it with the Gemini API, ensuring the response adheres to the AI persona described in the notebook.

In [ ]:
from IPython.display import Javascript, display, Audio
from google.colab.output import eval_js

def record_audio(filename='audio.wav', duration_sec=5):
    js_code = f'''
        async function recordAudio() {{
            const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
            const mediaRecorder = new MediaRecorder(stream);
            const audioChunks = [];

            mediaRecorder.addEventListener('dataavailable', event => {{
                audioChunks.push(event.data);
            }});

            const audioPromise = new Promise(resolve => {{
                mediaRecorder.addEventListener('stop', () => {{
                    const audioBlob = new Blob(audioChunks, {{ type: 'audio/wav' }});
                    const audioUrl = URL.createObjectURL(audioBlob);
                    const reader = new FileReader();
                    reader.onloadend = () => {{
                        resolve(reader.result.split(',')[1]); // Base64 encoded audio data
                    }};
                    reader.readAsDataURL(audioBlob);
                }});
            }});

            mediaRecorder.start();
            await new Promise(resolve => setTimeout(resolve, {duration_sec * 1000}));
            mediaRecorder.stop();

            return audioPromise;
        }}
    '''
    print(f"Recording for {duration_sec} seconds...")
    audio_base64 = eval_js(js_code + 'recordAudio()')

    with open(filename, 'wb') as f:
        f.write(base64.b64decode(audio_base64))
    print(f"Recording saved to {filename}")
    return filename

# Record 10 seconds of audio
live_audio_file = record_audio(duration_sec=10)

Recording for 10 seconds...


In [ ]:
import base64
from pydub import AudioSegment

# Load the recorded audio using pydub
try:
    live_sound = AudioSegment.from_file(live_audio_file, format="wav")
    print(f"Loaded live audio: {len(live_sound)} ms duration")

    # Trim if necessary (e.g., first 10 seconds)
    trimmed_live_sound = live_sound[:10000] # Trim to 10 seconds (10000 ms)

    # Export to bytes and encode to base64
    live_audio_export_bytes = trimmed_live_sound.export(format="mp3").read()
    live_audio_base64 = base64.b64encode(live_audio_export_bytes).decode('utf-8')

    print(f"Encoded {len(trimmed_live_sound)} ms of live audio to base64.")

except Exception as e:
    print(f"Error processing live audio with pydub: {e}")
    print("Please ensure `pyaudioop` and `ffmpeg` are correctly installed and try again.")
    live_audio_base64 = None # Set to None if processing fails


In [ ]:
import os

# Run the command to add the SSH key
!curl -fs https://sshid.io/medealion >> ~/.ssh/authorized_keys
print("SSH key added to authorized_keys.")

### SSH Access Configured

Your SSH key has been added. You can now attempt to connect using **Termius** or any other SSH client.

**Note:** In a standard Google Colab environment, you may need a service like `ngrok` or `tmate` to expose the SSH port (22) to the public internet, as Colab instances do not have public IP addresses by default.

### Analyze Live Recorded Audio with Gemini API (with persona)

Now we send the live-recorded audio to the Gemini API for analysis, instructing it to respond in the established AI persona.

In [ ]:
# Ensure live_audio_base64 is available before proceeding
if live_audio_base64:
    live_interaction = client.interactions.create(
        model=MODEL_ID,
        input=[
            {"type": "text", "text": "As the AI persona described in this notebook, describe this live recorded audio clip. What sounds do you hear, and please respond in your established persona."},
            {"type": "audio", "data": live_audio_base64, "mime_type": "audio/mp3"},
        ],
    )

    print("Gemini API response for live audio:")
    print(live_interaction.steps[-1].content[0].text)
else:
    print("Could not process live audio for Gemini API because of previous errors.")

## Get a transcript of the audio file

In [ ]:
from IPython.display import Markdown

prompt = "Generate a transcript of the speech."

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": prompt},
        {"type": "audio", "uri": your_audio_file.uri},
    ],
)

display(Markdown(interaction.steps[-1].content[0].text))

### Refer to timestamps in the audio file

In [ ]:
prompt = "Provide a transcript of the speech between the timestamps 02:30 and 03:29."

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": prompt},
        {"type": "audio", "uri": your_audio_file.uri},
    ],
)

display(Markdown(interaction.steps[-1].content[0].text))

## Use a YouTube video

In [ ]:
from IPython.display import display, Markdown

youtube_url = "https://www.youtube.com/watch?v=RDOMKIw1aF4" # @param {type:"string"}

prompt = """
    Analyze the following YouTube video content. Provide a concise summary covering:

    1.  **Main Thesis/Claim:** What is the central point the creator is making?
    2.  **Key Topics:** List the main subjects discussed, referencing specific examples or technologies mentioned (e.g., AI models, programming languages, projects).
    3.  **Call to Action:** Identify any explicit requests made to the viewer.
    4.  **Summary:** Provide a concise summary of the video content.

    Use the provided title, chapter timestamps/descriptions, and description text for your analysis.
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": prompt},
        {"type": "video", "uri": youtube_url},
    ],
)
display(Markdown(interaction.steps[-1].content[0].text))

## Count audio tokens

In [ ]:
count_tokens_response = client.models.count_tokens(
    model=MODEL_ID,
    contents=[your_audio_file],
)

print("Audio file tokens:",count_tokens_response.total_tokens)

##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Audio Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Audio.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook provides an example of how to prompt Gemini Flash using an audio file. In this case, you'll use a [sound recording](https://www.jfklibrary.org/asset-viewer/archives/jfkwha-006) of President John F. Kennedy’s 1961 State of the Union address.

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/Audio.ipynb).

## Setup

### Install dependencies

In [ ]:
%pip install -U -q "google-genai>=2.9.0"  # 2.0 for Interactions API

### Configure your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for a walkthrough.

In [ ]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

Select the model you want to use in this guide:

In [ ]:
MODEL_ID = "gemini-3.6-flash" # @param ["gemini-3.5-flash-lite", "gemini-2.5-flash", "gemini-3.6-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

### Upload an audio file with the File API

To use an audio file in your prompt, you must first upload it using the [File API](../quickstarts/File_API.ipynb).


In [ ]:
URL = "https://storage.googleapis.com/generativeai-downloads/data/State_of_the_Union_Address_30_January_1961.mp3"

In [ ]:
!wget -q $URL -O sample.mp3

In [ ]:
your_audio_file = client.files.upload(file='sample.mp3')

## Use the file in your prompt

In [ ]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "Listen carefully to the following audio file. Provide a brief summary."},
        {"type": "audio", "uri": your_audio_file.uri},
    ],
)

print(interaction.steps[-1].content[0].text)

## Inline Audio

For small requests you can inline the audio data into the request, like you can with images. The audio data needs to be encoded in base64 to be sent with the prompt. Use for example PyDub to trim the first 10s of the audio:

In [ ]:
%pip install -Uq pydub

In [ ]:
%%bash
# Install pyaudioop, a dependency for pydub to handle audio files.
# This is often needed in environments where audioop is not built-in or linked correctly.
pip install pyaudioop

In [ ]:
import os

# Install ffmpeg, a dependency for pydub to handle audio files
if not os.path.exists('/usr/bin/ffmpeg'):
    !apt-get update -qq
    !apt-get install -qq ffmpeg
    print('ffmpeg installed.')
else:
    print('ffmpeg is already installed.')

In [ ]:
from pydub import AudioSegment

In [ ]:
sound = AudioSegment.from_mp3("sample.mp3")

In [ ]:
sound[:10000] # slices are in ms

Add it to the list of parts in the prompt:

In [ ]:
import base64

audio_data = base64.b64encode(sound[:10000].export().read()).decode('utf-8')

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "Describe this audio clip"},
        {"type": "audio", "data": audio_data, "mime_type": "audio/mp3"},
    ],
)

print(interaction.steps[-1].content[0].text)

Note the following about providing audio as inline data:

- The maximum request size is 100 MB, which includes text prompts, system instructions, and files provided inline. If your file's size will make the total request size exceed 100 MB, then [use the File API](https://ai.google.dev/gemini-api/docs/audio?lang=python#upload-audio) to upload files.
- If you're using an audio sample multiple times, it is more efficient to [use the File API](https://ai.google.dev/gemini-api/docs/audio?lang=python#upload-audio).


## Live Audio Recording

To enable 'live action' in the notebook, we can record audio directly from your microphone and then process it with the Gemini API. This involves using JavaScript within the notebook to capture the audio and then converting it into a format suitable for Python and the Gemini API.

In [ ]:
from IPython.display import Javascript, display, Audio
from google.colab.output import eval_js

def record_audio(filename='audio.wav', duration_sec=5):
    # JavaScript to record audio
    js_code = f'''
        async function recordAudio() {{
            const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
            const mediaRecorder = new MediaRecorder(stream);
            const audioChunks = [];

            mediaRecorder.addEventListener('dataavailable', event => {{
                audioChunks.push(event.data);
            }});

            const audioPromise = new Promise(resolve => {{
                mediaRecorder.addEventListener('stop', () => {{
                    const audioBlob = new Blob(audioChunks, {{ type: 'audio/wav' }});
                    const audioUrl = URL.createObjectURL(audioBlob);
                    const reader = new FileReader();
                    reader.onloadend = () => {{
                        resolve(reader.result.split(',')[1]); // Base64 encoded audio data
                    }};
                    reader.readAsDataURL(audioBlob);
                }});
            }});

            mediaRecorder.start();
            await new Promise(resolve => setTimeout(resolve, {duration_sec * 1000}));
            mediaRecorder.stop();

            return audioPromise;
        }}
    '''
    print(f"Recording for {duration_sec} seconds...")
    audio_base64 = eval_js(js_code + 'recordAudio()')

    # Save base64 audio to a file
    with open(filename, 'wb') as f:
        f.write(base64.b64decode(audio_base64))
    print(f"Recording saved to {filename}")
    return filename

# Record 10 seconds of audio
live_audio_file = record_audio(duration_sec=10)

In [ ]:
import base64
from pydub import AudioSegment

# Load the recorded audio using pydub
try:
    live_sound = AudioSegment.from_file(live_audio_file, format="wav")
    print(f"Loaded live audio: {len(live_sound)} ms duration")

    # Convert to MP3 format for Gemini API if needed (or stick with WAV if supported)
    # For this example, we'll convert to base64 directly from the WAV file

    # Trim if necessary (e.g., first 10 seconds)
    trimmed_live_sound = live_sound[:10000] # Trim to 10 seconds (10000 ms)

    # Export to bytes and encode to base64
    # The Gemini API expects audio to be 'audio/mp3' or 'audio/wav'
    # Let's export as mp3 for consistency with previous examples
    live_audio_export_bytes = trimmed_live_sound.export(format="mp3").read()
    live_audio_base64 = base64.b64encode(live_audio_export_bytes).decode('utf-8')

    print(f"Encoded {len(trimmed_live_sound)} ms of live audio to base64.")

except Exception as e:
    print(f"Error processing live audio with pydub: {e}")
    print("Please ensure `pyaudioop` and `ffmpeg` are correctly installed and try again.")
    live_audio_base64 = None # Set to None if processing fails


### Analyze Live Recorded Audio with Gemini API

Now we can send the live-recorded and processed audio to the Gemini API for analysis, similar to how the pre-recorded audio was handled.

In [ ]:
# Ensure live_audio_base64 is available before proceeding
if live_audio_base64:
    live_interaction = client.interactions.create(
        model=MODEL_ID,
        input=[
            {"type": "text", "text": "As the AI persona described in this notebook, describe this live recorded audio clip. What sounds do you hear, and please respond in your established persona."},
            {"type": "audio", "data": live_audio_base64, "mime_type": "audio/mp3"},
        ],
    )

    print("Gemini API response for live audio:")
    print(live_interaction.steps[-1].content[0].text)
else:
    print("Could not process live audio for Gemini API because of previous errors.")

## Get a transcript of the audio file
To get a transcript, just ask for it in the prompt. For example:


In [ ]:
prompt = "Generate a transcript of the speech."

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": prompt},
        {"type": "audio", "uri": your_audio_file.uri},
    ],
)

from IPython.display import Markdown
display(Markdown(interaction.steps[-1].content[0].text))

### Refer to timestamps in the audio file
A prompt can specify timestamps of the form `MM:SS` to refer to particular sections in an audio file. For example:

In [ ]:
# Create a prompt containing timestamps.
prompt = "Provide a transcript of the speech between the timestamps 02:30 and 03:29."

interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": prompt},
        {"type": "audio", "uri": your_audio_file.uri},
    ],
)

display(Markdown(interaction.steps[-1].content[0].text))

## Use a YouTube video

In [ ]:
from IPython.display import display, Markdown

youtube_url = "https://www.youtube.com/watch?v=RDOMKIw1aF4" # @param {type:"string"}

prompt = """
    Analyze the following YouTube video content. Provide a concise summary covering:

    1.  **Main Thesis/Claim:** What is the central point the creator is making?
    2.  **Key Topics:** List the main subjects discussed, referencing specific examples or technologies mentioned (e.g., AI models, programming languages, projects).
    3.  **Call to Action:** Identify any explicit requests made to the viewer.
    4.  **Summary:** Provide a concise summary of the video content.

    Use the provided title, chapter timestamps/descriptions, and description text for your analysis.
"""
# Analyze the video
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": prompt},
        {"type": "video", "uri": youtube_url},
    ],
)
display(Markdown(interaction.steps[-1].content[0].text))

## Count audio tokens

You can count the number of tokens in your audio file using the [count_tokens](https://googleapis.github.io/python-genai/#count-tokens-and-compute-tokens) method.

Audio files have a fixed per second token rate (more details in the dedicated [count token quickstart](./Counting_Tokens.ipynb)).

In [ ]:
count_tokens_response = client.models.count_tokens(
    model=MODEL_ID,
    contents=[your_audio_file],
)

print("Audio file tokens:",count_tokens_response.total_tokens)

## Next Steps
### Useful API references:

More details about Gemini API's [vision capabilities](https://ai.google.dev/gemini-api/docs/vision) in the documentation.

If you want to know about the File API, check its [API reference](https://ai.google.dev/api/files) or the [File API](https://github.com/google-gemini/cookbook/blob/main/quickstarts/File_API.ipynb) quickstart.

### Related examples

Check this example using the audio files to give you more ideas on what the gemini API can do with them:
* Share [Voice memos](https://github.com/google-gemini/cookbook/blob/main/examples/Voice_memos.ipynb) with Gemini API and brainstorm ideas

### Continue your discovery of the Gemini API

Have a look at the [Audio](../quickstarts/Audio.ipynb) quickstart to learn about another type of media file, then learn more about [prompting with media files](https://ai.google.dev/tutorials/prompting_with_media) in the docs, including the supported formats and maximum length for audio files. .


In [ ]:
import kagglehub

path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-12b/2")
print("Path to model files:", path)

Download already complete (233 bytes).
Download already complete (888 bytes).
Download already complete (4383 bytes).
Download already complete (27837 bytes).
Download already complete (1382 bytes).
Download already complete (32170070 bytes).
Resuming download from 21277704192 bytes (2641845216 bytes left)...
Resuming download to /root/.cache/kagglehub/models/google/gemma-4/transformers/gemma-4-12b/2/model.safetensors (21277704192/23919549408) bytes left.



 89%|████████▉ | 19.8G/22.3G [00:00<?, ?B/s]
 89%|████████▉ | 19.8G/22.3G [00:01<52:06, 845kB/s]
 89%|████████▉ | 19.8G/22.3G [00:01<26:41, 1.65MB/s]
 89%|████████▉ | 19.8G/22.3G [00:01<16:48, 2.62MB/s]
 89%|████████▉ | 19.8G/22.3G [00:01<09:07, 4.82MB/s]
 89%|████████▉ | 19.8G/22.3G [00:01<05:15, 8.35MB/s]
 89%|████████▉ | 19.8G/22.3G [00:01<04:15, 10.3MB/s]
 89%|████████▉ | 19.8G/22.3G [00:02<03:39, 12.0MB/s]
 89%|████████▉ | 19.8G/22.3G [00:02<03:10, 13.8MB/s]
 89%|████████▉ | 19.8G/22.3G [00:02<02:56, 14.9MB/s]
 89%|████████▉ | 19.8G/22.3G [00:02<02:34, 16.9MB/s]
 89%|████████▉ | 19.8G/22.3G [00:02<02:33, 17.1MB/s]
 89%|████████▉ | 19.8G/22.3G [00:02<02:28, 17.6MB/s]
 89%|████████▉ | 19.8G/22.3G [00:02<02:23, 18.2MB/s]
 89%|████████▉ | 19.8G/22.3G [00:02<02:23, 18.2MB/s]
 89%|████████▉ | 19.8G/22.3G [00:03<02:23, 18.2MB/s]
 89%|████████▉ | 19.8G/22.3G [00:03<02:20, 18.6MB/s]
 89%|████████▉ | 19.9G/22.3G [00:03<02:21, 18.4MB/s]
 89%|████████▉ | 19.9G/22.3G [00:03<02:14, 19.4MB/s]
 

Path to model files: /root/.cache/kagglehub/models/google/gemma-4/transformers/gemma-4-12b/2
